# Fig. 2 Generalization Sweep: Partial Quickcheck

This notebook is the interactive quickcheck for the `nf_generalize_fig2` sweep. It is wired for `SAMPLE_LABEL=dpm50` by default, so it will inspect the 50-step DPM samples for UNet-64, UNet-128, and UNet-256 when those files exist.

It does four things:

1. audits which training tasks have final checkpoints,
2. audits which `dpm50` sample files exist,
3. plots one-point/P(k)/image diagnostics for whatever samples are already available,
4. consumes the offline PCA and SSCD tables to make the Fig. 2-style reproducibility/generalization plots.

Important: after new samples are generated, the notebook can show them in the quick sample diagnostics immediately, but the PCA/SSCD Fig. 2 curves will not include them until the analyzer tables are regenerated. After the UNet-256 `dpm50` sample array finishes, rerun:

```bash
cd /home/jiamingp/diffusion_models_repo
PCA_COMPONENTS=8192 PCA_FIT_MAX_SLICES=0 PCA_TARGET_VARIANCE=0.98 SAMPLE_LABEL=dpm50 \
  sbatch -A huterer2 --time=00:30:00 --mem=32gb scripts/slurm/analyze_nf_generalize_fig2_pca.sbatch
SAMPLE_LABEL=dpm50 sbatch -A huterer2 scripts/slurm/analyze_nf_generalize_fig2_sscd.sbatch
```

Then rerun this notebook. The table coverage audit below will explicitly say whether PCA/SSCD include `u256` rows.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter
import numpy as np
import pandas as pd
import yaml

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

SWEEP_NAME = 'nf_generalize_fig2'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
CHECKPOINT_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/saved_runs') / SWEEP_NAME
SAMPLE_ROOT = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUTPUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'quickcheck'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get('NF_FIG2_SEED', 123))
SAMPLE_LABEL = os.environ.get('NF_FIG2_SAMPLE_LABEL', 'dpm50')
MAX_GENERATED = int(os.environ.get('NF_FIG2_MAX_GENERATED', 512))
MAX_REAL_RAW_CUBES = int(os.environ.get('NF_FIG2_MAX_REAL_RAW_CUBES', 16))
PK_NBINS = int(os.environ.get('NF_FIG2_PK_NBINS', 30))

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 240,
    'font.size': 15,
    'axes.titlesize': 17,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 20,
    'axes.grid': False,
    'axes.linewidth': 1.2,
    'lines.linewidth': 2.8,
    'lines.markersize': 8,
})

ARCH_LABELS = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256'}
ARCH_STYLES = {
    'u64': {'color': '#00b81f', 'marker': '^'},
    'u128': {'color': '#e41a1c', 'marker': 'o'},
    'u256': {'color': '#1027e8', 'marker': 's'},
}
PAIR_STYLES = {
    ('u64', 'u128'): {'color': '#e41a1c', 'marker': 'o'},
    ('u64', 'u256'): {'color': '#1027e8', 'marker': 's'},
    ('u128', 'u256'): {'color': '#00b81f', 'marker': '^'},
}
def arch_label(arch: str) -> str:
    return ARCH_LABELS.get(str(arch), str(arch))


def _arch_sort_key(arch: str) -> int:
    m = re.search(r'(\d+)', str(arch))
    return int(m.group(1)) if m else 10_000


def pair_label(pair: str) -> str:
    parts = sorted(str(pair).split('_vs_'), key=_arch_sort_key)
    return ' vs '.join(arch_label(p) for p in parts)


def pair_style(pair: str) -> dict[str, Any]:
    parts = tuple(sorted(str(pair).split('_vs_'), key=_arch_sort_key))
    return PAIR_STYLES.get(parts, {'color': '#333333', 'marker': 'o'})


def dataset_tick_label(x: float, _pos: int | None = None) -> str:
    if not np.isfinite(x) or x <= 0:
        return ''
    k = math.log2(float(x))
    if abs(k - round(k)) < 1e-7:
        return rf'$2^{{{int(round(k))}}}$'
    return f'{x:g}'


def dataset_ticks(values: Any) -> list[float]:
    vals = sorted({int(v) for v in np.asarray(list(values), dtype=float) if np.isfinite(v) and v > 0})
    if not vals:
        return []
    ticks = []
    for v in vals:
        k = int(round(math.log2(v)))
        if k % 2 == 1 or v in (vals[0], vals[-1]):
            ticks.append(float(v))
    return ticks


def polish_axis(ax, *, no_grid: bool = True) -> None:
    if no_grid:
        ax.grid(False)
    ax.tick_params(axis='both', which='major', length=5, width=1.15, direction='out')
    ax.tick_params(axis='both', which='minor', length=3, width=0.9, direction='out')
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
        spine.set_color('#222222')


def style_dataset_axis(ax, values: Any | None = None, xlabel: str = 'Training set size') -> None:
    ax.set_xscale('log', base=2)
    if values is not None:
        ticks = dataset_ticks(values)
        if ticks:
            ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FuncFormatter(dataset_tick_label))
    ax.set_xlabel(xlabel)
    polish_axis(ax)


def add_regime_labels(ax, values: Any | None = None, *, y_mem: float = 0.13, y_gen: float = 0.13) -> None:
    vals = [] if values is None else [float(v) for v in np.asarray(list(values), dtype=float) if np.isfinite(v) and v > 0]
    if vals:
        ax.set_xlim(min(vals) / 1.15, max(vals) * 1.15)
    ax.text(0.24, y_mem, 'Memorization\nregime', ha='center', va='center', fontsize=13, transform=ax.transAxes)
    ax.text(0.82, y_gen, 'Generalization\nregime', ha='center', va='center', fontsize=13, transform=ax.transAxes)


def save_and_show(fig, path: Path) -> None:
    fig.savefig(path, dpi=240, bbox_inches='tight')
    print('wrote', path)
    plt.show()

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH)
print('checkpoint root:', CHECKPOINT_ROOT)
print('sample root:', SAMPLE_ROOT)
print('seed:', SEED)


In [ ]:
rows = json.loads(MANIFEST_PATH.read_text())
for task_id, row in enumerate(rows):
    row['task_id'] = task_id

manifest_df = pd.DataFrame(rows)
display(manifest_df[[
    'task_id', 'run_name', 'arch', 'dataset_tag', 'dataset_size',
    'epochs', 'steps_per_epoch', 'actual_updates', 'checkpoint_every_n_epochs',
    'n_train_simulations'
]])


## Training Checkpoint Audit

A run is treated as complete if its latest `checkpoint-epoch-*` directory is at least `epochs - 1` from the manifest. This works even if the directory name is zero-padded or not.


In [ ]:
EPOCH_RE = re.compile(r'checkpoint-epoch-(\d+)')

def checkpoint_epoch(path: Path) -> int | None:
    m = EPOCH_RE.search(path.name)
    return int(m.group(1)) if m else None

ckpt_rows = []
for row in rows:
    ckpt_dir = Path(row['checkpoint_dir'])
    ckpts = sorted(ckpt_dir.glob('checkpoint-epoch-*')) if ckpt_dir.exists() else []
    parsed = [(checkpoint_epoch(p), p) for p in ckpts]
    parsed = [(e, p) for e, p in parsed if e is not None]
    latest_epoch = max((e for e, _ in parsed), default=None)
    latest_path = next((p for e, p in parsed if e == latest_epoch), None) if latest_epoch is not None else None
    final_epoch = int(row['epochs']) - 1
    ckpt_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'expected_final_epoch': final_epoch,
        'latest_epoch': latest_epoch,
        'final_checkpoint': latest_epoch is not None and latest_epoch >= final_epoch,
        'n_checkpoints': len(parsed),
        'checkpoint_dir': str(ckpt_dir),
        'latest_checkpoint': str(latest_path) if latest_path else '',
    })

ckpt_df = pd.DataFrame(ckpt_rows).sort_values(['arch', 'dataset_size'])
display(ckpt_df)
print('checkpoint audit:', ckpt_df['final_checkpoint'].value_counts(dropna=False).to_dict())

completed_ids = ckpt_df.loc[ckpt_df['final_checkpoint'], 'task_id'].astype(int).tolist()
missing_ids = ckpt_df.loc[~ckpt_df['final_checkpoint'], 'task_id'].astype(int).tolist()
print('completed task ids:', completed_ids)
print('missing/pending task ids:', missing_ids)


## Training Loss Curves

This reads the latest `metrics_epoch_*.json` from each checkpoint directory and plots loss against optimizer updates, not just epoch number. That matters here because the small-
N runs have many more epochs but are configured to have roughly the same total optimizer-update budget.

In [ ]:
def metric_candidates(row: dict[str, Any]) -> list[Path]:
    root = Path(row['checkpoint_dir'])
    paths: list[Path] = []
    if root.exists():
        paths.extend(sorted(root.glob('metrics_epoch_*.json')))
        metrics_json = root / 'metrics.json'
        if metrics_json.exists():
            paths.append(metrics_json)
        for ckpt in sorted(root.glob('checkpoint-epoch-*')):
            paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def _flatten_numeric(values: Any) -> np.ndarray:
    if values is None:
        return np.asarray([], dtype=float)
    out: list[float] = []

    def visit(x: Any) -> None:
        if x is None:
            return
        if isinstance(x, dict):
            for key in ('loss', 'value', 'mean', 'avg'):
                if key in x:
                    visit(x[key])
                    return
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for item in x:
                visit(item)
            return
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return

    visit(values)
    return np.asarray(out, dtype=float)


def read_latest_metrics(row: dict[str, Any]) -> tuple[dict[str, Any], Path | None]:
    paths = metric_candidates(row)
    if not paths:
        return {}, None
    def score(path: Path) -> tuple[int, float]:
        epoch = checkpoint_epoch(path) if 'metrics_epoch_' in path.name else -1
        return (epoch if epoch is not None else -1, path.stat().st_mtime)
    latest = max(paths, key=score)
    with latest.open() as f:
        return json.load(f), latest


def downsample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) <= max_points:
        return x, y
    idx = np.linspace(0, len(x) - 1, max_points, dtype=int)
    return x[idx], y[idx]

loss_by_run: dict[str, dict[str, Any]] = {}
loss_rows = []
for row in rows:
    metrics, metrics_path = read_latest_metrics(row)
    epoch_loss = _flatten_numeric(metrics.get('epoch_loss'))
    batch_loss = _flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
    epoch_lr = _flatten_numeric(metrics.get('epoch_lr', metrics.get('lr')))
    loss_by_run[row['run_name']] = {
        'metrics': metrics,
        'metrics_path': metrics_path,
        'epoch_loss': epoch_loss,
        'batch_loss': batch_loss,
        'epoch_lr': epoch_lr,
    }
    loss_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'steps_per_epoch': int(row.get('steps_per_epoch', 1)),
        'metrics_path': str(metrics_path) if metrics_path else None,
        'n_epoch_loss': len(epoch_loss),
        'final_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        'best_epoch_loss': float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
        'n_batch_loss': len(batch_loss),
        'final_batch_loss': float(batch_loss[-1]) if len(batch_loss) else np.nan,
    })

loss_df = pd.DataFrame(loss_rows).sort_values(['arch', 'dataset_size'])
display(loss_df)

if loss_df['n_epoch_loss'].sum() == 0 and loss_df['n_batch_loss'].sum() == 0:
    print('No training metrics JSON found yet.')
else:
    for arch, sub_rows in pd.DataFrame(rows).sort_values('dataset_size').groupby('arch'):
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
        for _, row in sub_rows.iterrows():
            run_name = row['run_name']
            info = loss_by_run.get(run_name, {})
            label = f"N={int(row['dataset_size'])}"
            steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1)))
            epoch_loss = np.asarray(info.get('epoch_loss', []), dtype=float)
            if len(epoch_loss):
                x = np.arange(len(epoch_loss), dtype=float) * steps_per_epoch
                x, y = downsample_xy(x, epoch_loss)
                axes[0].plot(x, y, lw=1.4, label=label)
            batch_loss = np.asarray(info.get('batch_loss', []), dtype=float)
            if len(batch_loss):
                x = np.arange(len(batch_loss), dtype=float)
                y = batch_loss
                if len(y) > 0:
                    window = max(1, len(y) // 1200)
                    if window > 1:
                        kernel = np.ones(window, dtype=float) / window
                        y = np.convolve(y, kernel, mode='valid')
                        x = x[:len(y)] + 0.5 * (window - 1)
                x, y = downsample_xy(x, y)
                axes[1].plot(x, y, lw=1.2, alpha=0.85, label=label)
            epoch_lr = np.asarray(info.get('epoch_lr', []), dtype=float)
            if len(epoch_lr):
                x = np.arange(len(epoch_lr), dtype=float) * steps_per_epoch
                x, y = downsample_xy(x, epoch_lr)
                axes[2].plot(x, y, lw=1.2, label=label)

        axes[0].set_title('epoch loss')
        axes[0].set_xlabel('optimizer update')
        axes[0].set_ylabel('mean training loss')
        axes[1].set_title('batch loss, smoothed')
        axes[1].set_xlabel('optimizer update')
        axes[1].set_ylabel('training loss')
        axes[2].set_title('learning rate')
        axes[2].set_xlabel('optimizer update')
        axes[2].set_ylabel('LR')
        for ax in axes:
            ax.grid(alpha=0.25)
            ax.set_yscale('log')
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.10), ncol=min(5, len(labels)))
        fig.suptitle(f'{arch}: Fig.2 sweep training curves')
        fig.tight_layout(rect=(0, 0.12, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_training_curves.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()

## Sample Audit and Commands

Run sampling only for task IDs with completed final checkpoints. If maintenance blocks long training jobs, sampling should still be short, but the scheduler may still reserve nodes depending on the maintenance window.


In [ ]:
def sample_path_for(row: dict[str, Any], seed: int = SEED, sample_label: str = SAMPLE_LABEL) -> Path:
    if row.get('sample_path'):
        raw = str(row['sample_path'])
        if '{sample_label}' not in raw:
            raw = raw.replace('raw_train_full', sample_label)
        return PROJECT_DIR / raw.format(seed=seed, sample_label=sample_label)
    return SAMPLE_ROOT / f"{row['run_name']}_seed{seed}_{sample_label}.npz"


def npz_n(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        with np.load(path) as data:
            key = 'samples' if 'samples' in data.files else data.files[0]
            return int(data[key].shape[0])
    except Exception as exc:
        print('failed reading', path, exc)
        return 0

sample_rows = []
for row in rows:
    path = sample_path_for(row)
    n = npz_n(path)
    sample_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'final_checkpoint': bool(ckpt_df.set_index('task_id').loc[row['task_id'], 'final_checkpoint']),
        'n_available': n,
        'status': 'ok' if n > 0 else 'missing',
        'sample_path': str(path),
    })

sample_df = pd.DataFrame(sample_rows).sort_values(['arch', 'dataset_size'])
display(sample_df)
print('sample audit:', sample_df['status'].value_counts().to_dict())
if 'arch' in sample_df and 'u256' in set(sample_df['arch'].astype(str)):
    u256_df = sample_df[sample_df['arch'].astype(str) == 'u256']
    n_ready = int((u256_df['n_available'] > 0).sum())
    print(f'u256 DPM50 sample coverage: {n_ready}/{len(u256_df)} runs have samples')

to_sample = sample_df[(sample_df['final_checkpoint']) & (sample_df['n_available'] == 0)]['task_id'].astype(int).tolist()
if to_sample:
    ids = ','.join(map(str, to_sample))
    print('Submit samples for completed checkpoints:')
    print(f'cd {PROJECT_DIR}')
    print(f'SAMPLE_LABEL={SAMPLE_LABEL} SAMPLER_CLASS=DPMSolverMultistepScheduler SAMPLER_STEPS=50 OVERWRITE=1 sbatch -A huterer2 --array={ids}%2 scripts/slurm/sample_nf_generalize_fig2_array.sbatch')
else:
    print('No completed unsampled tasks found.')


## Load Available Samples

This loads only sample files that already exist. Real data is capped with `MAX_REAL_RAW_CUBES` so the notebook stays responsive; the offline SSCD script should be used for the full-reference memorization/generalization score.


In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        return as_nchw(np.asarray(data[key], dtype=np.float32))


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=int)
    return arr[idx].copy()

loaded = {}
load_rows = []
for row in rows:
    path = sample_path_for(row)
    if not path.exists():
        continue
    config_path = PROJECT_DIR / row['config']
    generated = evenly_limit(load_npz_array(path), MAX_GENERATED)
    raw_cap = min(int(row.get('n_train_simulations', MAX_REAL_RAW_CUBES)), MAX_REAL_RAW_CUBES)
    real = as_nchw(load_real_from_config(config_path, max_raw_samples=raw_cap))
    loaded[row['run_name']] = {
        'spec': row,
        'real': real,
        'generated': generated,
        'sample_path': path,
    }
    load_rows.append({
        'task_id': row['task_id'],
        'run_name': row['run_name'],
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'n_real_loaded': len(real),
        'raw_cap': raw_cap,
        'n_generated': len(generated),
        'sample_path': str(path),
    })

loaded_df = pd.DataFrame(load_rows).sort_values(['arch', 'dataset_size']) if load_rows else pd.DataFrame()
display(loaded_df)
print('loaded sample rows:', len(loaded))


## Available Sample Quality Metrics

These are quick-check metrics using capped real references. They are useful for catching broken runs or broad quality trends, not for final ranking. Lower `hist_l1` and `pk_log10_mae` are better; `std_ratio` and P(k) ratios near 1 are better.


In [ ]:
metric_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    real = bundle['real']
    generated = bundle['generated']
    rh = field_histogram(real, bins=120)
    gh = field_histogram(generated, bins=120)
    edges = np.asarray(rh['bin_edges'])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh['hist']) - np.asarray(gh['hist']))) * width)
    metric_rows.append({
        'task_id': row['task_id'],
        'run_name': run_name,
        'arch': row['arch'],
        'dataset_size': int(row['dataset_size']),
        'n_real': len(real),
        'n_generated': len(generated),
        'hist_l1': hist_l1,
        'generated_std': gh['std'],
        'real_std': rh['std'],
        'std_ratio': gh['std'] / max(rh['std'], 1e-30),
        **power_spectrum_summary(real, generated, nbins=PK_NBINS),
    })

metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    metrics_df = metrics_df.sort_values(['arch', 'dataset_size'])
    out = OUTPUT_DIR / 'nf_generalize_fig2_partial_metrics.csv'
    metrics_df.to_csv(out, index=False)
    print('wrote', out)
    display(metrics_df)
else:
    print('No loaded samples yet.')


In [ ]:

def plot_arch_metric_summary(metrics_df: pd.DataFrame, arch: str) -> None:
    sub = metrics_df[metrics_df['arch'] == arch].sort_values('dataset_size')
    if sub.empty:
        print(f'No loaded {arch_label(arch)} samples for the quick metric summary.')
        return
    x = sub['dataset_size'].astype(float).to_numpy()
    style = ARCH_STYLES.get(arch, {'color': '#333333', 'marker': 'o'})
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8), sharex=True, constrained_layout=True)
    panels = [
        ('hist_l1', 'One-point L1 error', 'lower is better'),
        ('pk_log10_mae', r'$P(k)$ log10 MAE', 'lower is better'),
        ('std_ratio', 'Generated / real std', 'target = 1'),
    ]
    for ax, (col, ylabel, title) in zip(axes, panels):
        ax.plot(x, sub[col], color=style['color'], marker=style['marker'], label=arch_label(arch))
        if col == 'std_ratio':
            ax.axhline(1.0, color='black', ls=':', lw=1.6)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        style_dataset_axis(ax, x)
    axes[0].legend(frameon=True, loc='best')
    fig.suptitle(f'{arch_label(arch)} sample-quality quick check ({SAMPLE_LABEL})')
    out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_{SAMPLE_LABEL}_partial_metrics.png'
    save_and_show(fig, out)


if len(metrics_df):
    for arch in sorted(metrics_df['arch'].unique(), key=_arch_sort_key):
        plot_arch_metric_summary(metrics_df, arch)
else:
    print('No loaded samples yet.')


## Detailed One-point and P(k) Comparisons

The summary table above gives scalar errors. This block shows the actual one-point histogram overlays and mean power-spectrum ratios for a small set of available dataset sizes. Use `NF_FIG2_DETAIL_TAGS=d2p06,d2p10,d2p15` before launching Jupyter if you want fewer/more panels.

In [ ]:

DETAIL_TAGS = [x.strip() for x in os.environ.get('NF_FIG2_DETAIL_TAGS', 'd2p08,d2p15').split(',') if x.strip()]
POSTER_FIDELITY_ARCH = os.environ.get('NF_FIG2_POSTER_FIDELITY_ARCH', 'u128')

if loaded:
    for arch in sorted({bundle['spec']['arch'] for bundle in loaded.values()}):
        bundles = [b for b in loaded.values() if b['spec']['arch'] == arch and b['spec'].get('dataset_tag') in DETAIL_TAGS]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
        if not bundles:
            print(f'No loaded rows for {arch} and detail tags {DETAIL_TAGS}')
            continue
        n = len(bundles)
        with plt.rc_context({
            'font.family': 'serif',
            'font.serif': ['DejaVu Serif'],
            'mathtext.fontset': 'dejavuserif',
            'font.size': 17,
            'axes.titlesize': 20,
            'axes.labelsize': 19,
            'xtick.labelsize': 15,
            'ytick.labelsize': 15,
            'legend.fontsize': 15,
            'figure.titlesize': 22,
        }):
            fig, axes = plt.subplots(2, n, figsize=(max(4.2 * n, 8.6), 7.6), squeeze=False, constrained_layout=True)
            for col, bundle in enumerate(bundles):
                row = bundle['spec']
                real = bundle['real']
                generated = bundle['generated']

                rh = field_histogram(real, bins=140)
                gh = field_histogram(generated, bins=140)
                edges = np.asarray(rh['bin_edges'])
                centers = 0.5 * (edges[:-1] + edges[1:])
                axes[0, col].plot(centers, rh['hist'], color='black', lw=2.4, label='real')
                axes[0, col].plot(centers, gh['hist'], color='#0072B2', lw=2.2, label='generated')
                axes[0, col].set_yscale('log')
                axes[0, col].set_title(rf"$N_{{2D}}={int(row['dataset_size']):,}$ one-point")
                axes[0, col].set_xlabel('Normalized field value')
                if col == 0:
                    axes[0, col].set_ylabel('Pixel PDF')
                    axes[0, col].legend(frameon=False, loc='upper right')

                pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
                pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
                ratio = np.nanmean(pk_gen, axis=0) / np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
                axes[1, col].plot(kbins, ratio, marker='o', ms=4.8, lw=2.1, color='#0072B2')
                axes[1, col].axhline(1.0, color='0.25', ls='--', lw=1.5)
                axes[1, col].set_ylim(0, max(2.0, float(np.nanquantile(ratio, 0.98)) * 1.15 if np.isfinite(ratio).any() else 2.0))
                axes[1, col].set_title('Mean $P(k)$ ratio')
                axes[1, col].set_xlabel('$k$ bin')
                if col == 0:
                    axes[1, col].set_ylabel('generated / real')
                for ax in axes[:, col]:
                    polish_axis(ax)
                    ax.grid(False)
            fig.suptitle(f'{arch_label(arch)} physical-statistics check', y=1.02)
            out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_detail_hist_pk.png'
            save_and_show(fig, out)
            if arch == POSTER_FIDELITY_ARCH:
                poster_out = OUTPUT_DIR / 'fidelity_onepoint_pk.png'
                fig.savefig(poster_out, dpi=300, bbox_inches='tight')
                print('wrote', poster_out)
else:
    print('No loaded samples available for detailed one-point/P(k) plots.')



## Optional: Physical Fidelity Across Training Checkpoints

Nick suggested using this as a learning-process view: keep the training set fixed, then compare generated samples from early/intermediate/final checkpoints. The cell below looks for epoch-snapshot `.npz` files created by `scripts/slurm/sample_nf_generalize_epoch_snapshots.sbatch`. If they are missing, it prints the command to run on Great Lakes instead of failing.


In [ ]:

EPOCH_SNAPSHOT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'epoch_snapshots'
EPOCH_SNAPSHOT_RUN = os.environ.get('NF_FIG2_EPOCH_SNAPSHOT_RUN', 'nf_fig2_u128_d2p15_noaug_200k')
EPOCH_SNAPSHOT_LABEL = os.environ.get('NF_FIG2_EPOCH_SNAPSHOT_LABEL', SAMPLE_LABEL)
EPOCH_FIDELITY_MAX_FILES = int(os.environ.get('NF_FIG2_EPOCH_FIDELITY_MAX_FILES', 4))


def load_npz_image_stack(path: Path) -> np.ndarray:
    data = np.load(path)
    key = next((k for k in ('samples', 'images', 'arr_0') if k in data), data.files[0])
    arr = np.asarray(data[key])
    if arr.ndim == 4:
        if arr.shape[1] in (1, 3):
            arr = arr[:, 0]
        elif arr.shape[-1] in (1, 3):
            arr = arr[..., 0]
    if arr.ndim != 3:
        raise ValueError(f'Expected (N,H,W) image stack in {path}, got {arr.shape}')
    return arr.astype(np.float32, copy=False)


def epoch_from_snapshot(path: Path) -> int:
    m = re.search(r'_epoch(\d+)_', path.name)
    if not m:
        raise ValueError(f'Could not parse epoch from {path.name}')
    return int(m.group(1))

snapshot_files = sorted(
    EPOCH_SNAPSHOT_DIR.glob(f'{EPOCH_SNAPSHOT_RUN}_epoch*_seed*_{EPOCH_SNAPSHOT_LABEL}.npz'),
    key=epoch_from_snapshot,
)
if len(snapshot_files) > EPOCH_FIDELITY_MAX_FILES:
    keep = np.linspace(0, len(snapshot_files) - 1, EPOCH_FIDELITY_MAX_FILES, dtype=int)
    snapshot_files = [snapshot_files[i] for i in keep]

ref_bundle = next((b for b in loaded.values() if b['spec']['run_name'] == EPOCH_SNAPSHOT_RUN), None)
if not snapshot_files:
    print('No epoch snapshot files found for:', EPOCH_SNAPSHOT_RUN)
    print('Run this on Great Lakes, then rerun the notebook:')
    print('  cd /home/jiamingp/diffusion_models_repo')
    print('  RUN_NAME=%s OVERWRITE=1 sbatch -A huterer2 scripts/slurm/sample_nf_generalize_epoch_snapshots.sbatch' % EPOCH_SNAPSHOT_RUN)
elif ref_bundle is None:
    print('Found snapshot files, but the matching real reference bundle is not loaded:', EPOCH_SNAPSHOT_RUN)
else:
    real = ref_bundle['real']
    pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
    mean_pk_real = np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
    rh = field_histogram(real, bins=140)
    centers = 0.5 * (np.asarray(rh['bin_edges'])[:-1] + np.asarray(rh['bin_edges'])[1:])

    colors = plt.cm.viridis(np.linspace(0.12, 0.88, len(snapshot_files)))
    with plt.rc_context({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'font.size': 17,
        'axes.titlesize': 20,
        'axes.labelsize': 19,
        'xtick.labelsize': 15,
        'ytick.labelsize': 15,
        'legend.fontsize': 14,
        'savefig.dpi': 300,
    }):
        fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.6), constrained_layout=True)
        axes[0].plot(centers, rh['hist'], color='black', lw=2.4, label='real')
        axes[1].axhline(1.0, color='0.25', ls='--', lw=1.5)

        for color, path in zip(colors, snapshot_files):
            epoch = epoch_from_snapshot(path)
            samples = load_npz_image_stack(path)
            gh = field_histogram(samples, bins=rh['bin_edges'])
            axes[0].plot(centers, gh['hist'], color=color, lw=2.0, label=f'epoch {epoch}')

            pk_gen, _ = batch_power_spectra(samples[:, None, :, :], nbins=PK_NBINS)
            ratio = np.nanmean(pk_gen, axis=0) / mean_pk_real
            axes[1].plot(kbins, ratio, color=color, marker='o', ms=4.5, lw=2.0, label=f'epoch {epoch}')

        axes[0].set_yscale('log')
        axes[0].set_xlabel('Normalized field value')
        axes[0].set_ylabel('Pixel PDF')
        axes[0].set_title('One-point PDF')
        axes[1].set_xlabel('$k$ bin')
        axes[1].set_ylabel('generated / real')
        axes[1].set_ylim(0, 2.0)
        axes[1].set_title('Mean $P(k)$ ratio')
        for ax in axes:
            polish_axis(ax)
            ax.grid(False)
        axes[0].legend(frameon=False, loc='best')
        axes[1].legend(frameon=False, loc='best')
        fig.suptitle(f'Physical fidelity during training: {EPOCH_SNAPSHOT_RUN}', y=1.04)

    out = OUTPUT_DIR / 'nf_generalize_fig2_epoch_fidelity_onepoint_pk.png'
    fig.savefig(out, bbox_inches='tight')
    print('wrote', out)
    plt.show()


## Real vs Generated Image Panels

For each available run, the top row is a capped real reference slice and the bottom row is a generated sample. This is the fastest way to see whether the low-N models are producing off-manifold artifacts, memorized-looking structures, or plausible fields.


In [ ]:
if loaded:
    for arch in sorted({bundle['spec']['arch'] for bundle in loaded.values()}):
        bundles = [b for b in loaded.values() if b['spec']['arch'] == arch]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
        n = len(bundles)
        fig, axes = plt.subplots(2, n, figsize=(max(3*n, 8), 5.4), squeeze=False)
        for col, bundle in enumerate(bundles):
            row = bundle['spec']
            real = bundle['real']
            generated = bundle['generated']
            vmin = float(np.nanquantile(real, 0.005))
            vmax = float(np.nanquantile(real, 0.995))
            axes[0, col].imshow(real[0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[0, col].set_title(f"N={row['dataset_size']} real")
            axes[1, col].imshow(generated[0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[1, col].set_title('generated')
            for ax in axes[:, col]:
                ax.set_xticks([])
                ax.set_yticks([])
        fig.suptitle(f'{arch}: available real vs generated examples')
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_partial_images.png'
        fig.savefig(out, dpi=180, bbox_inches='tight')
        print('wrote', out)
        plt.show()
else:
    print('No samples available to plot yet.')


## Generated Samples Versus Closest Loaded Training Slices

This is a quick visual copy check. For each available run it takes a generated sample, searches the loaded training-slice subset for the closest map in pixel MSE, and plots generated versus closest training slices. For the poster version, the high-$N$ columns are hidden and the difference row is optional so the labels stay readable. The search is capped to keep the notebook responsive; the full SSCD job is still the serious all-reference nearest-neighbor analysis.

In [ ]:

NN_MAX_REAL = int(os.environ.get('NF_FIG2_NN_MAX_REAL', 2048))
NN_MAX_GENERATED = int(os.environ.get('NF_FIG2_NN_MAX_GENERATED', 8))
NN_DISPLAY_GENERATED_INDEX = int(os.environ.get('NF_FIG2_NN_DISPLAY_GENERATED_INDEX', 0))
# Poster view: keep only 2^6 ... 2^11 so the image panels stay readable.
NN_POSTER_MAX_DATASET_SIZE = int(os.environ.get('NF_FIG2_NN_POSTER_MAX_DATASET_SIZE', 2**11))
NN_POSTER_SHOW_DIFF = os.environ.get('NF_FIG2_NN_POSTER_SHOW_DIFF', '0') == '1'


def closest_training_for_generated(
    generated: np.ndarray,
    real: np.ndarray,
    max_generated: int = NN_MAX_GENERATED,
    max_real: int = NN_MAX_REAL,
    real_chunk: int = 256,
) -> pd.DataFrame:
    gen = as_nchw(generated)[:max_generated].astype(np.float32, copy=False)
    ref = as_nchw(real)[:max_real].astype(np.float32, copy=False)
    gen_flat = gen.reshape(len(gen), -1).astype(np.float32, copy=False)
    ref_flat = ref.reshape(len(ref), -1).astype(np.float32, copy=False)
    gen_norm = np.linalg.norm(gen_flat, axis=1)
    ref_norm = np.linalg.norm(ref_flat, axis=1)
    rows_out = []
    for gi, g in enumerate(gen_flat):
        best_mse = np.inf
        best_idx = -1
        for start in range(0, len(ref_flat), real_chunk):
            chunk = ref_flat[start:start + real_chunk]
            diff = chunk - g[None, :]
            mse = np.mean(diff * diff, axis=1)
            local = int(np.argmin(mse))
            if float(mse[local]) < best_mse:
                best_mse = float(mse[local])
                best_idx = start + local
        denom = max(float(gen_norm[gi] * ref_norm[best_idx]), 1e-30)
        cos = float(np.dot(g, ref_flat[best_idx]) / denom)
        rows_out.append({
            'generated_index': gi,
            'nearest_real_index': best_idx,
            'nearest_mse': best_mse,
            'nearest_rmse': float(np.sqrt(best_mse)),
            'nearest_cosine': cos,
        })
    return pd.DataFrame(rows_out)

nn_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    nn = closest_training_for_generated(bundle['generated'], bundle['real'])
    for record in nn.to_dict('records'):
        nn_rows.append({
            'task_id': row['task_id'],
            'run_name': run_name,
            'arch': row['arch'],
            'dataset_size': int(row['dataset_size']),
            'n_real_searched': min(len(bundle['real']), NN_MAX_REAL),
            'n_generated_searched': min(len(bundle['generated']), NN_MAX_GENERATED),
            **record,
        })

nn_df = pd.DataFrame(nn_rows)
if len(nn_df):
    nn_df = nn_df.sort_values(['arch', 'dataset_size', 'generated_index'])
    out = OUTPUT_DIR / 'nf_generalize_fig2_partial_pixel_nn.csv'
    nn_df.to_csv(out, index=False)
    print('wrote', out)
    display(nn_df)
else:
    print('No loaded samples available for nearest-training comparison.')

def dataset_size_label(n: int) -> str:
    n = int(n)
    log2n = np.log2(n)
    if np.isfinite(log2n) and abs(log2n - round(log2n)) < 1e-9:
        return rf'$2^{{{int(round(log2n))}}}$'
    return f'{n:,}'


def plot_generated_vs_pixel_nn_for_arch(arch: str, *, max_dataset_size: int = NN_POSTER_MAX_DATASET_SIZE) -> None:
    bundles = [
        b for b in loaded.values()
        if b['spec']['arch'] == arch and int(b['spec']['dataset_size']) <= max_dataset_size
    ]
    bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
    plot_records = []
    for bundle in bundles:
        row = bundle['spec']
        sub = nn_df[(nn_df['run_name'] == row['run_name']) & (nn_df['generated_index'] == NN_DISPLAY_GENERATED_INDEX)]
        if len(sub) == 0:
            sub = nn_df[nn_df['run_name'] == row['run_name']].head(1)
        if len(sub) == 0:
            continue
        rec = sub.iloc[0]
        gi = int(rec['generated_index'])
        ri = int(rec['nearest_real_index'])
        gen_img = bundle['generated'][gi, 0]
        real_img = bundle['real'][ri, 0]
        plot_records.append({
            'bundle': bundle,
            'spec': row,
            'rec': rec,
            'gen_img': gen_img,
            'real_img': real_img,
            'diff': np.abs(gen_img - real_img),
        })
    n = len(plot_records)
    if n == 0:
        print(f'No loaded rows for {arch} with N <= {max_dataset_size}.')
        return

    field_values = np.concatenate([img.ravel() for rec in plot_records for img in (rec['gen_img'], rec['real_img'])])
    diff_values = np.concatenate([rec['diff'].ravel() for rec in plot_records])
    vmin = float(np.nanquantile(field_values, 0.005))
    vmax = float(np.nanquantile(field_values, 0.995))
    diff_vmax = float(np.nanquantile(diff_values, 0.995))

    nrows = 3 if NN_POSTER_SHOW_DIFF else 2
    row_height = 3.05 if nrows == 2 else 2.55
    with plt.rc_context({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'font.size': 23,
        'axes.titlesize': 26,
        'axes.labelsize': 28,
        'figure.titlesize': 29,
        'savefig.dpi': 300,
    }):
        fig, axes = plt.subplots(
            nrows, n,
            figsize=(max(2.85 * n + 2.3, 12.5), row_height * nrows + 1.40),
            squeeze=False,
        )
        fig.subplots_adjust(left=0.135, right=0.995, bottom=0.18, top=0.84, wspace=0.055, hspace=0.10)

        for col, item in enumerate(plot_records):
            row = item['spec']
            rec = item['rec']
            axes[0, col].imshow(item['gen_img'], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[0, col].set_title(dataset_size_label(row['dataset_size']), pad=10, fontweight='bold', fontsize=27)
            axes[1, col].imshow(item['real_img'], cmap='viridis', vmin=vmin, vmax=vmax)
            axes[1, col].set_title(f"cos={rec['nearest_cosine']:.2f}", pad=7, fontsize=19)
            if NN_POSTER_SHOW_DIFF:
                axes[2, col].imshow(item['diff'], cmap='magma', vmin=0.0, vmax=diff_vmax)
            for ax in axes[:, col]:
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_linewidth(0.9)
                    spine.set_color('#333333')

        row_labels = ['Generated', 'Closest training'] + (['Absolute difference'] if NN_POSTER_SHOW_DIFF else [])
        for rr, label in enumerate(row_labels):
            axes[rr, 0].set_ylabel(
                label,
                rotation=45,
                ha='right',
                va='center',
                labelpad=70,
                fontweight='bold',
                fontsize=26,
            )

        arch_name = arch_label(arch) if 'arch_label' in globals() else arch
        fig.suptitle(f'{arch_name}: generated samples and nearest training slices', y=0.975, fontweight='bold', fontsize=29)

        arrow_ax = fig.add_axes([0.22, 0.045, 0.66, 0.07])
        arrow_ax.axis('off')
        arrow_ax.annotate(
            '',
            xy=(0.98, 0.45),
            xytext=(0.02, 0.45),
            arrowprops=dict(arrowstyle='-|>', lw=2.2, color='0.25', shrinkA=0, shrinkB=0),
        )
        arrow_ax.text(0.50, 0.70, 'bigger training set', ha='center', va='bottom', fontsize=23, color='0.25')

    out = OUTPUT_DIR / f'nf_generalize_fig2_{arch}_generated_vs_pixel_nn.png'
    fig.savefig(out, bbox_inches='tight')
    print('wrote', out)
    plt.show()

if len(nn_df):
    for arch in sorted(nn_df['arch'].unique(), key=_arch_sort_key):
        plot_generated_vs_pixel_nn_for_arch(arch)


## DPM50 Sample-Quality Summaries By Architecture

This compact panel is shown before the feature-space reproducibility/generalizability score so the raw sample behavior is visible first. It now plots every architecture available in `metrics_df`, including `UNet-256` after the `u256` DPM50 samples are generated.


In [ ]:
if len(metrics_df):
    available_arches = sorted(metrics_df['arch'].dropna().astype(str).unique(), key=_arch_sort_key)
    print('sample-quality architectures:', [arch_label(a) for a in available_arches])
    if 'u256' not in set(available_arches):
        print('UNet-256 is not in metrics_df yet; check the sample audit above and rerun this notebook after u256 DPM50 samples are present.')
    for arch in available_arches:
        plot_arch_metric_summary(metrics_df, arch)
else:
    print('No sample metrics loaded, so the architecture quality summaries cannot be plotted yet.')


## Paper Fig. 2 Style Reproducibility and Generalizability

This section reproduces the structure of Fig. 2 from the paper using the runs currently available. The key statistic is computed per generated sample:

\[
s_j = \max_i \mathrm{sim}(x_j, y_i), \qquad \mathrm{GL}(\tau)=1-\Pr[s_j > \tau].
\]

There are two different threshold choices shown here. The fixed-`tau` plot uses one hand-chosen similarity cutoff, currently `tau=0.9`, so it is mainly a sensitivity/reference plot. The adaptive plots choose `tau` from the real training set itself. For each architecture and dataset size, the notebook uses the leave-one-out train-real nearest-neighbor similarities:

\[
s^{\rm train}_j = \max_{i \ne j} \mathrm{sim}(y_j, y_i), \qquad \tau_{q}=\mathrm{quantile}_{q}(s^{\rm train}).
\]

The adaptive q95/q99 scores then apply `tau_q95` or `tau_q99` to generated samples. This is not chosen from generated samples, because that would be circular: if a model memorizes, the generated distribution would push the threshold upward and hide the copy behavior. For reproducibility between two architectures, the code uses the stricter of the two corresponding train-real thresholds at that `N` (the larger tau), so a pair only counts as reproducible if it is closer than what real training maps usually are to their own nearest real neighbor.

Interpretation: q95 is a looser threshold and gives a smoother score; q99 is stricter and closer to a copy/memorization criterion. PCA rank/explained variance defines the feature basis, but `tau` is a cosine-similarity cutoff inside that basis, so PCA and SSCD should be calibrated separately.

The notebook therefore plots both versions: fixed-`tau` panels as a sensitivity check, and adaptive q95/q99 Fig. 2-style panels as the preferred threshold-calibrated reproducibility/generalization view.


In [ ]:

TABLE_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'tables'
PCA_METRICS_PATH = TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'
PCA_RP_PATH = TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_reproducibility.csv'
PCA_MODE_PATH = TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_mode_norms.csv'
SSCD_METRICS_PATH = TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'
SSCD_RP_PATH = TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_reproducibility.csv'
FIG2_PRIMARY_TAU = float(os.environ.get('NF_FIG2_PRIMARY_TAU', 0.9))
FIG2_TAU_SUFFIX = f"{FIG2_PRIMARY_TAU:.3f}".rstrip('0').rstrip('.').replace('.', 'p')
PLOT_ADAPTIVE = os.environ.get('NF_FIG2_PLOT_ADAPTIVE', '1') == '1'


def read_table(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        if 'dataset_size' in df:
            df = df.sort_values(['arch', 'dataset_size'] if 'arch' in df else ['dataset_size'])
        return df
    return pd.DataFrame()


def show_table_summary(name: str, df: pd.DataFrame, path: Path) -> None:
    print(name, 'loaded' if len(df) else 'missing', path)
    if df.empty:
        return
    keep = [
        'arch', 'arch_pair', 'dataset_tag', 'dataset_size', 'n_generated',
        f'gen_gl_fixed_{FIG2_TAU_SUFFIX}', f'rp_unpaired_fixed_{FIG2_TAU_SUFFIX}',
        f'rp_fixed_{FIG2_TAU_SUFFIX}', 'threshold_q95', 'threshold_q99',
        'gen_gl_q95', 'gen_gl_q99', 'rp_threshold_q95', 'rp_threshold_q99',
        'rp_q95', 'rp_q99', 'rp_unpaired_q95', 'rp_unpaired_q99',
        'pca_rank', 'pca_explained_variance_sum',
    ]
    keep = [c for c in keep if c in df.columns]
    display(df[keep] if keep else df.head())


EXPECTED_FIG2_ARCHES = ('u64', 'u128', 'u256')
EXPECTED_FIG2_PAIRS = ('u64_vs_u128', 'u64_vs_u256', 'u128_vs_u256')


def canonical_arch_pair(pair: str) -> str:
    parts = sorted(str(pair).split('_vs_'), key=_arch_sort_key)
    return '_vs_'.join(parts)


def show_analysis_coverage(name: str, metrics: pd.DataFrame, rp: pd.DataFrame) -> None:
    if metrics.empty:
        print(f'{name} coverage: missing metrics table')
        return
    if 'arch' in metrics and 'dataset_size' in metrics:
        arch_counts = (
            metrics.assign(arch=metrics['arch'].astype(str))
            .groupby('arch')['dataset_size']
            .nunique()
            .reindex(EXPECTED_FIG2_ARCHES, fill_value=0)
            .astype(int)
            .to_dict()
        )
        print(f'{name} metric coverage by architecture:', arch_counts)
        missing_arches = [arch for arch, count in arch_counts.items() if count == 0]
        if missing_arches:
            print(f'{name}: missing {missing_arches}; rerun DPM50 sampling and then the {name} analyzer before trusting the Fig. 2 curves.')
    if not rp.empty and 'arch_pair' in rp and 'dataset_size' in rp:
        pair_counts = (
            rp.assign(arch_pair=rp['arch_pair'].map(canonical_arch_pair))
            .groupby('arch_pair')['dataset_size']
            .nunique()
            .reindex(EXPECTED_FIG2_PAIRS, fill_value=0)
            .astype(int)
            .to_dict()
        )
        print(f'{name} reproducibility coverage by pair:', pair_counts)
        missing_pairs = [pair for pair, count in pair_counts.items() if count == 0]
        if missing_pairs:
            print(f'{name}: missing reproducibility pairs {missing_pairs}; this usually means the analyzer table was made before all architectures were sampled.')


def plot_pca_mode_norm_distribution(path: Path = PCA_MODE_PATH) -> None:
    if not path.exists():
        print('PCA mode norm table missing:', path)
        print('Rerun the PCA analyzer after pulling the latest code, e.g.:')
        print('  SAMPLE_LABEL=dpm50 sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_fig2_pca.sbatch')
        return
    modes = pd.read_csv(path).sort_values('component')
    if modes.empty:
        print('PCA mode norm table is empty:', path)
        return
    component = modes['component'].to_numpy(dtype=float)
    strength = modes['sqrt_explained_variance_ratio'].to_numpy(dtype=float)
    cumulative = modes['cumulative_explained_variance'].to_numpy(dtype=float)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
    axes[0].plot(component, strength, color='#3b4cc0', lw=2.4)
    axes[0].set_yscale('log')
    if len(component) > 64:
        axes[0].set_xscale('log')
    axes[0].set_xlabel('PCA mode index')
    axes[0].set_ylabel(r'Mode strength $\sqrt{\lambda_i / \sum_j \lambda_j}$')
    axes[0].set_title('PCA mode strength')

    axes[1].plot(component, cumulative, color='#b40426', lw=2.8)
    axes[1].axhline(0.98, color='black', ls=':', lw=1.8)
    axes[1].set_ylim(0, 1.02)
    axes[1].set_xlabel('PCA modes retained')
    axes[1].set_ylabel('Cumulative explained variance')
    axes[1].set_title('Variance captured')

    bins = min(60, max(12, int(np.sqrt(len(strength)))))
    axes[2].hist(strength[np.isfinite(strength) & (strength > 0)], bins=bins, color='#4c78a8', alpha=0.85)
    axes[2].set_yscale('log')
    axes[2].set_xlabel(r'PCA mode strength $\sqrt{\lambda_i / \sum_j \lambda_j}$')
    axes[2].set_ylabel('Mode count')
    axes[2].set_title('Mode-strength distribution')

    for ax in axes:
        polish_axis(ax)
    rank = int(modes['pca_rank'].iloc[0]) if 'pca_rank' in modes else len(modes)
    ev = float(modes['pca_explained_variance_sum'].iloc[0]) if 'pca_explained_variance_sum' in modes else float(cumulative[-1])
    fig.suptitle(f'PCA basis diagnostics: {rank:,} modes, explained variance={ev:.3f} ({100 * ev:.1f}%)')
    out = OUTPUT_DIR / 'nf_generalize_fig2_pca_mode_norm_distribution.png'
    save_and_show(fig, out)


def plot_fig2_style(metrics: pd.DataFrame, rp: pd.DataFrame, *, feature_name: str, tau_suffix: str = FIG2_TAU_SUFFIX) -> None:
    if metrics.empty:
        print(f'Missing {feature_name} metrics table.')
        return
    gl_col = f'gen_gl_fixed_{tau_suffix}'
    paired_rp_col = f'rp_fixed_{tau_suffix}'
    unpaired_rp_col = f'rp_unpaired_fixed_{tau_suffix}'
    if gl_col not in metrics.columns:
        print(f'{feature_name}: missing {gl_col}; available GL columns:', [c for c in metrics.columns if c.startswith('gen_gl_fixed_')])
        return

    x_values = list(metrics['dataset_size'].astype(float))
    if not rp.empty and 'dataset_size' in rp:
        x_values.extend(rp['dataset_size'].astype(float).tolist())

    fig, axes = plt.subplots(1, 2, figsize=(16.5, 5.8), sharex=False, constrained_layout=True)

    rp_col = unpaired_rp_col if (not rp.empty and unpaired_rp_col in rp.columns) else paired_rp_col
    rp_label = 'nearest-neighbor' if rp_col == unpaired_rp_col else 'same-index'
    if not rp.empty and rp_col in rp.columns:
        for pair, sub in rp.groupby('arch_pair', sort=False):
            sub = sub.sort_values('dataset_size')
            style = pair_style(pair)
            axes[0].plot(
                sub['dataset_size'], sub[rp_col],
                color=style['color'], marker=style['marker'], lw=3.0, ms=8.5,
                label=pair_label(pair),
            )
    else:
        axes[0].text(0.5, 0.5, 'reproducibility table missing', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_ylabel('Reproducibility')
    axes[0].set_title(f'Reproducibility ({rp_label}, tau={FIG2_PRIMARY_TAU:g})')

    for arch, sub in metrics.groupby('arch', sort=False):
        sub = sub.sort_values('dataset_size')
        style = ARCH_STYLES.get(arch, {'color': '#333333', 'marker': 'o'})
        axes[1].plot(
            sub['dataset_size'], sub[gl_col],
            color=style['color'], marker=style['marker'], lw=3.0, ms=8.5,
            label=arch_label(arch),
        )
    axes[1].set_ylabel('Generalization')
    axes[1].set_title(f'Generalization (tau={FIG2_PRIMARY_TAU:g})')

    for ax, y_text in zip(axes, (0.14, 0.16)):
        add_regime_labels(ax, x_values, y_mem=y_text, y_gen=y_text)
        ax.set_ylim(-0.04, 1.04)
        style_dataset_axis(ax, x_values)
        ax.legend(frameon=True, loc='best')
    title = f'{feature_name} fixed-threshold memorization and generalization'
    if feature_name.lower() == 'pca':
        suffix = pca_basis_suffix(metrics)
        if suffix:
            print('PCA basis used for Fig. 2 plot:' + suffix.replace('\n', ' '))
            title += suffix
    fig.suptitle(title)
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_paper_fig2_attempt.png'
    save_and_show(fig, out)


def pca_basis_suffix(metrics: pd.DataFrame) -> str:
    if {'pca_rank', 'pca_explained_variance_sum'}.issubset(metrics.columns):
        ranks = pd.to_numeric(metrics['pca_rank'], errors='coerce').dropna().unique()
        evs = pd.to_numeric(metrics['pca_explained_variance_sum'], errors='coerce').dropna().unique()
        if len(ranks) and len(evs):
            rank = int(ranks[0])
            ev = float(evs[0])
            basis_kind = '98% variance basis' if ev >= 0.98 else 'low-rank diagnostic basis'
            return f'\nPCA basis: {rank:,} modes, explained variance={ev:.3f} ({100 * ev:.1f}%; {basis_kind})'
    return ''


def plot_fig2_adaptive_style(metrics: pd.DataFrame, rp: pd.DataFrame, *, feature_name: str, quantile: str = 'q99') -> None:
    if metrics.empty:
        print(f'Missing {feature_name} metrics table for adaptive {quantile} plot.')
        return
    gl_col = f'gen_gl_{quantile}'
    rp_col = f'rp_unpaired_{quantile}' if (not rp.empty and f'rp_unpaired_{quantile}' in rp.columns) else f'rp_{quantile}'
    threshold_col = f'threshold_{quantile}'
    rp_threshold_col = f'rp_threshold_{quantile}'
    if gl_col not in metrics.columns:
        print(f'{feature_name}: missing {gl_col}; adaptive GL columns:', [c for c in metrics.columns if c.startswith('gen_gl_q')])
        return
    if rp.empty or rp_col not in rp.columns:
        print(f'{feature_name}: missing {rp_col}; adaptive RP columns:', [c for c in rp.columns if c.startswith('rp')])
        return

    x_values = list(metrics['dataset_size'].astype(float))
    if 'dataset_size' in rp:
        x_values.extend(rp['dataset_size'].astype(float).tolist())

    fig, axes = plt.subplots(1, 2, figsize=(16.5, 5.8), sharex=False, constrained_layout=True)

    for pair, sub in rp.groupby('arch_pair', sort=False):
        sub = sub.sort_values('dataset_size')
        style = pair_style(pair)
        axes[0].plot(
            sub['dataset_size'], sub[rp_col],
            color=style['color'], marker=style['marker'], lw=3.0, ms=8.5,
            label=pair_label(pair),
        )
    axes[0].set_ylabel('Reproducibility')
    axes[0].set_title(f'Generated-to-generated reproducibility\ncutoff = train-real tau_{quantile}')

    for arch, sub in metrics.groupby('arch', sort=False):
        sub = sub.sort_values('dataset_size')
        style = ARCH_STYLES.get(arch, {'color': '#333333', 'marker': 'o'})
        axes[1].plot(
            sub['dataset_size'], sub[gl_col],
            color=style['color'], marker=style['marker'], lw=3.0, ms=8.5,
            label=arch_label(arch),
        )
    axes[1].set_ylabel('Generalization')
    axes[1].set_title(f'Generated-to-training-real generalization\ncutoff = train-real tau_{quantile}')

    for ax, y_text in zip(axes, (0.14, 0.16)):
        add_regime_labels(ax, x_values, y_mem=y_text, y_gen=y_text)
        ax.set_ylim(-0.04, 1.04)
        style_dataset_axis(ax, x_values)
        ax.legend(frameon=True, loc='best')

    note = ''
    if threshold_col in metrics.columns:
        finite = pd.to_numeric(metrics[threshold_col], errors='coerce').dropna()
        if len(finite):
            note += f'\ntrain-real tau_{quantile} range: {finite.min():.3f}-{finite.max():.3f}'
    if rp_threshold_col in rp.columns:
        finite = pd.to_numeric(rp[rp_threshold_col], errors='coerce').dropna()
        if len(finite):
            note += f'; RP stricter-pair tau range: {finite.min():.3f}-{finite.max():.3f}'
    title = f'{feature_name} adaptive-threshold memorization and generalization ({quantile}){note}'
    if feature_name.lower() == 'pca':
        title += pca_basis_suffix(metrics)
    fig.suptitle(title)
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_adaptive_{quantile}_paper_fig2_attempt.png'
    save_and_show(fig, out)


def fixed_tau_columns(df: pd.DataFrame, prefix: str) -> list[tuple[float, str]]:
    cols = []
    for col in df.columns:
        if not col.startswith(prefix):
            continue
        raw = col[len(prefix):]
        try:
            tau = float(raw.replace('p', '.'))
        except ValueError:
            continue
        cols.append((tau, col))
    return sorted(cols)


def dataset_label(size: float) -> str:
    if size > 0:
        power = np.log2(float(size))
        if np.isclose(power, round(power), atol=1e-6):
            return rf'$2^{{{int(round(power))}}}$'
    return f'{int(size):,}'


def plot_tau_calibration(metrics: pd.DataFrame, rp: pd.DataFrame, *, feature_name: str) -> None:
    if metrics.empty:
        print(f'{feature_name}: no metrics table for tau calibration.')
        return
    threshold_cols = [c for c in ('threshold_q95', 'threshold_q99') if c in metrics.columns]
    rp_fixed = fixed_tau_columns(rp, 'rp_unpaired_fixed_') if not rp.empty else []
    if not rp_fixed and not rp.empty:
        rp_fixed = fixed_tau_columns(rp, 'rp_fixed_')
    gl_fixed = fixed_tau_columns(metrics, 'gen_gl_fixed_')
    if not threshold_cols and not rp_fixed and not gl_fixed:
        print(f'{feature_name}: no threshold columns available for tau calibration.')
        return

    fig, axes = plt.subplots(1, 2, figsize=(16.5, 5.6), constrained_layout=True)

    if threshold_cols:
        for arch, sub in metrics.groupby('arch', sort=False):
            sub = sub.sort_values('dataset_size')
            style = ARCH_STYLES.get(arch, {'color': '#333333', 'marker': 'o'})
            for col, ls, label_suffix in [('threshold_q95', '--', 'train-real q95'), ('threshold_q99', '-', 'train-real q99')]:
                if col not in sub:
                    continue
                axes[0].plot(
                    sub['dataset_size'], sub[col],
                    color=style['color'], marker=style['marker'], ls=ls, lw=2.5, ms=7.5,
                    label=f'{arch_label(arch)} {label_suffix}',
                )
        style_dataset_axis(axes[0], metrics['dataset_size'])
        axes[0].set_ylabel('Similarity threshold')
        axes[0].set_title('Empirical tau from train-real nearest neighbors')
        axes[0].legend(frameon=True, fontsize=9, ncol=2)
    else:
        axes[0].text(0.5, 0.5, 'missing train-real q95/q99 thresholds', ha='center', va='center', transform=axes[0].transAxes)

    focus_size = None
    if not rp.empty and 'dataset_size' in rp and len(rp_fixed):
        focus_size = float(pd.to_numeric(rp['dataset_size'], errors='coerce').dropna().max())
    elif len(gl_fixed):
        focus_size = float(pd.to_numeric(metrics['dataset_size'], errors='coerce').dropna().max())

    if focus_size is not None:
        rp_focus = rp[np.isclose(pd.to_numeric(rp.get('dataset_size', np.nan), errors='coerce'), focus_size)] if not rp.empty else pd.DataFrame()
        metrics_focus = metrics[np.isclose(pd.to_numeric(metrics['dataset_size'], errors='coerce'), focus_size)]
        band_values = []
        for col in ('rp_threshold_q95', 'rp_threshold_q99'):
            if col in rp_focus:
                band_values.extend(pd.to_numeric(rp_focus[col], errors='coerce').dropna().tolist())
        for col in ('threshold_q95', 'threshold_q99'):
            if col in metrics_focus:
                band_values.extend(pd.to_numeric(metrics_focus[col], errors='coerce').dropna().tolist())
        band_values = [float(x) for x in band_values if np.isfinite(x)]
        if band_values:
            axes[1].axvspan(min(band_values), max(band_values), color='#f6c1b6', alpha=0.35, lw=0, label='train-real q95-q99 band')

        for pair, sub in rp_focus.groupby('arch_pair', sort=False):
            if not len(rp_fixed):
                continue
            y = [float(pd.to_numeric(sub[col], errors='coerce').mean()) for _tau, col in rp_fixed]
            x = [tau for tau, _col in rp_fixed]
            style = pair_style(pair)
            axes[1].plot(x, y, color=style['color'], marker=style['marker'], lw=2.8, ms=8, label=f'{pair_label(pair)} RP')

        for arch, sub in metrics_focus.groupby('arch', sort=False):
            if not len(gl_fixed):
                continue
            y = [float(pd.to_numeric(sub[col], errors='coerce').mean()) for _tau, col in gl_fixed]
            x = [tau for tau, _col in gl_fixed]
            style = ARCH_STYLES.get(arch, {'color': '#333333', 'marker': 'o'})
            axes[1].plot(x, y, color=style['color'], marker=style['marker'], ls=':', lw=2.6, ms=7, label=f'{arch_label(arch)} GL')

        axes[1].axvline(FIG2_PRIMARY_TAU, color='black', lw=1.8, ls='--', label=f'current tau={FIG2_PRIMARY_TAU:g}')
        axes[1].set_ylim(-0.04, 1.04)
        axes[1].set_xlabel('Fixed similarity threshold tau')
        axes[1].set_ylabel('Score at selected tau')
        axes[1].set_title(f'Fixed-threshold sensitivity at N={dataset_label(focus_size)}')
        axes[1].legend(frameon=True, fontsize=9, ncol=2)
    else:
        axes[1].text(0.5, 0.5, 'missing fixed-threshold columns', ha='center', va='center', transform=axes[1].transAxes)

    for ax in axes:
        polish_axis(ax)
    fig.suptitle(f'{feature_name}: tau calibration in similarity space')
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_tau_calibration.png'
    save_and_show(fig, out)


def plot_adaptive_rp(rp: pd.DataFrame, *, feature_name: str) -> None:
    if rp.empty:
        return
    cols = [c for c in ('rp_q95', 'rp_q99') if c in rp.columns]
    if not cols:
        return
    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
    for pair, sub in rp.groupby('arch_pair', sort=False):
        sub = sub.sort_values('dataset_size')
        style = pair_style(pair)
        for col in cols:
            ax.plot(sub['dataset_size'], sub[col], marker=style['marker'], color=style['color'], lw=2.4, label=f'{pair_label(pair)} {col}')
    ax.set_ylim(-0.04, 1.04)
    style_dataset_axis(ax, rp['dataset_size'])
    ax.set_ylabel('Gen-gen fraction above train-real tau')
    ax.set_title(f'{feature_name}: adaptive RP, tau from train-real q95/q99')
    ax.legend(frameon=True, fontsize=10)
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_adaptive_rp.png'
    save_and_show(fig, out)


def plot_adaptive_gl(metrics: pd.DataFrame, *, feature_name: str) -> None:
    if metrics.empty:
        return
    cols = [c for c in ('gen_gl_q95', 'gen_gl_q99') if c in metrics.columns]
    if not cols:
        return
    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
    for arch, sub in metrics.groupby('arch', sort=False):
        sub = sub.sort_values('dataset_size')
        style = ARCH_STYLES.get(arch, {'color': '#333333', 'marker': 'o'})
        for col in cols:
            ax.plot(sub['dataset_size'], sub[col], marker=style['marker'], color=style['color'], lw=2.4, label=f'{arch_label(arch)} {col}')
    ax.set_ylim(-0.04, 1.04)
    style_dataset_axis(ax, metrics['dataset_size'])
    ax.set_ylabel('Gen-real GL = 1 - fraction above train-real tau')
    ax.set_title(f'{feature_name}: adaptive GL, tau from train-real q95/q99')
    ax.legend(frameon=True, fontsize=10)
    out = OUTPUT_DIR / f'nf_generalize_fig2_{feature_name.lower()}_adaptive_gl.png'
    save_and_show(fig, out)


pca_fig2_df = read_table(PCA_METRICS_PATH)
pca_rp_df = read_table(PCA_RP_PATH)
sscd_fig2_df = read_table(SSCD_METRICS_PATH)
sscd_rp_df = read_table(SSCD_RP_PATH)

for name, df, path in [
    ('PCA metrics', pca_fig2_df, PCA_METRICS_PATH),
    ('PCA reproducibility', pca_rp_df, PCA_RP_PATH),
    ('SSCD metrics', sscd_fig2_df, SSCD_METRICS_PATH),
    ('SSCD reproducibility', sscd_rp_df, SSCD_RP_PATH),
]:
    show_table_summary(name, df, path)

show_analysis_coverage('PCA', pca_fig2_df, pca_rp_df)
show_analysis_coverage('SSCD', sscd_fig2_df, sscd_rp_df)

plot_pca_mode_norm_distribution(PCA_MODE_PATH)
plot_tau_calibration(pca_fig2_df, pca_rp_df, feature_name='PCA')
plot_tau_calibration(sscd_fig2_df, sscd_rp_df, feature_name='SSCD')
plot_fig2_style(pca_fig2_df, pca_rp_df, feature_name='PCA')
plot_fig2_style(sscd_fig2_df, sscd_rp_df, feature_name='SSCD')
for q in ('q95', 'q99'):
    plot_fig2_adaptive_style(pca_fig2_df, pca_rp_df, feature_name='PCA', quantile=q)
    plot_fig2_adaptive_style(sscd_fig2_df, sscd_rp_df, feature_name='SSCD', quantile=q)

if PLOT_ADAPTIVE:
    plot_adaptive_rp(pca_rp_df, feature_name='PCA')
    plot_adaptive_gl(pca_fig2_df, feature_name='PCA')
    plot_adaptive_rp(sscd_rp_df, feature_name='SSCD')
    plot_adaptive_gl(sscd_fig2_df, feature_name='SSCD')


## Poster Figure: Generalization Only

This cell makes the clean single-panel PCA version for the poster. The default uses the adaptive q95 threshold: a generated sample counts as generalized if it is not unusually close to the training set compared with the real-data nearest-neighbor baseline. q95 is the main operating point because it is stable and easy to read; q99 and fixed tau are kept as sensitivity checks, not as the main poster result.


In [ ]:
import matplotlib.patheffects as pe

POSTER_ARCH_ORDER = ['u64', 'u128', 'u256']

POSTER_ARCH_STYLE = {
    # Okabe-Ito colorblind-safe palette.
    'u64':  {'label': 'UNet-64',  'color': '#009E73', 'marker': '^'},
    'u128': {'label': 'UNet-128', 'color': '#D55E00', 'marker': 'o'},
    'u256': {'label': 'UNet-256', 'color': '#0072B2', 'marker': 's'},
}

def fixed_tau_suffix(tau: float) -> str:
    return f"{float(tau):.3f}".rstrip('0').rstrip('.').replace('.', 'p')

def choose_generalization_column(
    metrics: pd.DataFrame,
    *,
    score_mode: str = 'adaptive',
    quantile: str = 'q95',
    tau: float = 0.9,
) -> str:
    if score_mode == 'adaptive':
        col = f'gen_gl_{quantile}'
    elif score_mode == 'fixed':
        col = f'gen_gl_fixed_{fixed_tau_suffix(tau)}'
    else:
        raise ValueError("score_mode must be 'adaptive' or 'fixed'")
    if col not in metrics.columns:
        available = [c for c in metrics.columns if c.startswith('gen_gl')]
        raise KeyError(f'Missing column {col}. Available generalization columns: {available}')
    return col

def plot_poster_pca_generalization(
    metrics: pd.DataFrame,
    *,
    score_mode: str = 'adaptive',
    quantile: str = 'q95',
    tau: float = 0.9,
    title: str = 'Generated fields become less training-set-like with more data',
    out_name: str | None = None,
) -> None:
    gl_col = choose_generalization_column(
        metrics,
        score_mode=score_mode,
        quantile=quantile,
        tau=tau,
    )

    # Use Matplotlib's bundled serif/math fonts for a stable LaTeX-like poster style.
    with plt.rc_context({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'font.size': 20,
        'axes.labelsize': 24,
        'axes.titlesize': 25,
        'xtick.labelsize': 20,
        'ytick.labelsize': 20,
        'legend.fontsize': 21,
        'figure.dpi': 140,
        'savefig.dpi': 300,
    }):
        fig, ax = plt.subplots(figsize=(8.9, 6.15))
        fig.subplots_adjust(top=0.75, bottom=0.18, left=0.14, right=0.985)

        all_x: list[float] = []
        for arch in POSTER_ARCH_ORDER:
            sub = metrics[metrics['arch'].astype(str) == arch].sort_values('dataset_size')
            if sub.empty:
                continue

            x = sub['dataset_size'].astype(float).to_numpy()
            y = sub[gl_col].astype(float).to_numpy()
            all_x.extend(x.tolist())

            st = POSTER_ARCH_STYLE[arch]
            ax.plot(
                x,
                y,
                color=st['color'],
                marker=st['marker'],
                lw=4.2,
                ms=10.7,
                markeredgecolor='white',
                markeredgewidth=1.1,
                solid_capstyle='round',
                label=st['label'],
            )

        if not all_x:
            raise ValueError('No plotted points found for the requested architectures.')

        ax.set_xscale('log', base=2)
        ax.set_ylim(-0.03, 1.06)
        ax.set_xlim(min(all_x) / 1.16, max(all_x) * 1.16)

        ticks = sorted(set(int(x) for x in all_x))
        ax.set_xticks(ticks)
        ax.set_xticklabels([rf'$2^{{{int(np.log2(t))}}}$' for t in ticks])

        ax.set_xlabel('Training set size', labelpad=8)
        ax.set_ylabel('Generalization score', labelpad=10)
        ax.set_title(title, pad=48, fontweight='normal')

        ax.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.35)
        ax.spines['bottom'].set_linewidth(1.35)
        ax.tick_params(axis='both', width=1.25, length=6)

        # Labels are intentionally outside the steep transitions so they do not obscure data.
        text_effect = [pe.withStroke(linewidth=4.0, foreground='white')]
        label_y = 0.30
        ax.text(
            2**6.15,
            label_y,
            'memorization\nregime',
            ha='left',
            va='center',
            fontsize=21,
            color='0.25',
            linespacing=0.95,
            path_effects=text_effect,
        )
        ax.text(
            2**13.15,
            label_y,
            'generalization\nregime',
            ha='center',
            va='center',
            fontsize=21,
            color='0.25',
            linespacing=0.95,
            path_effects=text_effect,
        )

        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, 1.22),
            ncol=3,
            frameon=False,
            handlelength=2.1,
            columnspacing=1.35,
        )

        if out_name is None:
            if score_mode == 'adaptive':
                out_name = f'nf_generalize_fig2_poster_pca_generalization_{quantile}_serif_clean.png'
            else:
                out_name = f'nf_generalize_fig2_poster_pca_generalization_fixed_tau_{fixed_tau_suffix(tau)}_serif_clean.png'

        out = OUTPUT_DIR / out_name
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

# Main poster version.
plot_poster_pca_generalization(pca_fig2_df, score_mode='adaptive', quantile='q95')

# Sensitivity checks.
# plot_poster_pca_generalization(pca_fig2_df, score_mode='adaptive', quantile='q99')
# plot_poster_pca_generalization(pca_fig2_df, score_mode='fixed', tau=0.9)
